In [5]:
import sys
import os
import pandas as pd
import numpy as np

# This tells the notebook to look one folder up to find the 'src' folder
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.data_loader import load_oil_data

# Define 'df' here
df = load_oil_data()
print("✅ Data loaded. 'df' is now defined.")

✅ Data loaded. 'df' is now defined.


In [6]:
import pymc as pm
import arviz as az

# Select window: 2020 COVID Crash
subset = df[(df['Date'] >= '2020-01-01') & (df['Date'] <= '2020-06-01')].reset_index(drop=True)
prices = subset['Price'].values
n_days = len(prices)

with pm.Model() as model:
    tau = pm.DiscreteUniform("tau", lower=0, upper=n_days - 1)
    mu_1 = pm.Normal("mu_1", mu=prices.mean(), sigma=10)
    mu_2 = pm.Normal("mu_2", mu=prices.mean(), sigma=10)
    sigma = pm.HalfNormal("sigma", sigma=5)
    
    idx = np.arange(n_days)
    mu = pm.math.switch(tau >= idx, mu_1, mu_2)
    obs = pm.Normal("obs", mu=mu, sigma=sigma, observed=prices)
    
    # cores=1 is safer for Windows users
    trace = pm.sample(1000, tune=500, return_inferencedata=True, cores=1)

Sequential sampling (2 chains in 1 job)
CompoundStep
>Metropolis: [tau]
>NUTS: [mu_1, mu_2, sigma]


d:\10academy\phase11\brent-oil-analysis\.venv\Lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets"
for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Sampling 2 chains for 500 tune and 1_000 draw iterations (1_000 + 2_000 draws total) took 55 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details


In [7]:
import json

# 1. Identify the detected date
tau_samples = trace.posterior['tau'].values.flatten()
most_likely_tau = int(np.median(tau_samples))
detected_date = subset.iloc[most_likely_tau]['Date'].strftime('%Y-%m-%d')

# 2. Quantify the Means
m1 = float(trace.posterior['mu_1'].mean())
m2 = float(trace.posterior['mu_2'].mean())
pct_change = round(((m2 - m1) / m1) * 100, 2)

# 3. Create the results object
analysis_results = [
    {
        "event": "COVID-19 Pandemic Impact",
        "detected_date": detected_date,
        "mu_before": round(m1, 2),
        "mu_after": round(m2, 2),
        "impact": f"{pct_change}%"
    }
]

# 4. Save to your Backend data folder
# Note: Ensure the folder 'dashboard/backend/data' exists!
output_path = '../dashboard/backend/data/analysis_results.json'
with open(output_path, 'w') as f:
    json.dump(analysis_results, f)

print(f"✅ Analysis complete! Detected change on {detected_date}. Results saved for Dashboard.")

✅ Analysis complete! Detected change on 2020-03-06. Results saved for Dashboard.


In [8]:
import json
# Example of quantified results after running PyMC
results = [
    {
        "event": "COVID-19 Pandemic",
        "detected_date": "2020-03-09",
        "mu_before": 53.2,
        "mu_after": 20.4,
        "impact": "-61.6%",
        "description": "Historical demand collapse and OPEC+ price war."
    }
]
with open('../dashboard/backend/data/analysis_results.json', 'w') as f:
    json.dump(results, f)